In [ ]:
import argparse
import logging
import os
import random
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from pathlib import Path
from torch import optim
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

import wandb
from evaluate import evaluate
from unet import UNet
from utils.data_loading import BasicDataset, CarvanaDataset
from utils.dice_score import dice_loss

In [ ]:
model = UNet(n_channels=3, n_classes=2, bilinear=False)

In [ ]:
print(  f'Network:\n'
        f'\t{model.n_channels} input channels\n'
        f'\t{model.n_classes} output channels (classes)\n'
        f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

In [ ]:
dir_img = Path('./data/imgs/')
dir_mask = Path('./data/masks/')
dir_checkpoint = Path('./checkpoints/')
img_scale: float = 0.5
dataset = CarvanaDataset(dir_img, dir_mask, img_scale)
val_percent: float = 0.1
n_val = int(len(dataset) * val_percent)
n_train = len(dataset) - n_val
train_set, val_set = random_split(dataset, [n_train, n_val], generator=torch.Generator().manual_seed(0))
batch_size: int = 1
loader_args = dict(batch_size=batch_size, num_workers=0, pin_memory=True)
train_loader = DataLoader(train_set, shuffle=True, **loader_args)

在正常的图片训练中，image size 为 Image shape: torch.Size([1, 3, 640, 959])。
- batchsize
- channel number
- width
- height

In [ ]:
images, true_masks = dataset[0]['image'], dataset[0]['mask']

In [ ]:
images.shape

直接从 dataset 中取出的图片，image size 为 torch.Size([3, 640, 959])
- channel number
- width
- height

In [ ]:
images = images.unsqueeze(0)
images.shape[1] == model.n_channels

device = 'cpu'
images = images.to(device=device, dtype=torch.float32, memory_format=torch.channels_last)

masks_pred = model(images)

masks_pred.shape

---

# `nn.Conv1d()` test

In [ ]:
import torch
import torch.nn as nn
x = torch.arange(1, 50).float()
x = x.view(1, 1, -1)
conv1d = nn.Conv1d(in_channels=1, out_channels=1, kernel_size=7, bias=False)
conv1d.weight = nn.Parameter(torch.tensor([[[1., 1., 1., 1., 1., 1., 1.]]]))
output = conv1d(x)
print(output)

---

# `DoubleConv_1D()` test

In [ ]:
from unet.unet_parts import *
x = torch.arange(1, 365).float()
x = x.view(1, 1, -1)
n_channels = 1
inc = (DoubleConv_1D(n_channels, 32))
x1 = inc(x)
print(x.shape, x1.shape)

In [ ]:
down1 = (Down_1D(32, 64))
down2 = (Down_1D(64, 128))
down3 = (Down_1D(128, 256))

In [ ]:
x2 = down1(x1)
x3 = down2(x2)
x4 = down3(x3)
x2.shape, x3.shape, x4.shape

In [ ]:
up1 = (Up_1D(256, 128 // 1, False))
up2 = (Up_1D(128, 64 // 1, False))
up3 = (Up_1D(64, 32 // 1, False))

In [ ]:
x = up1(x4, x3)
print(x.size())
x = up2(x, x2)
print(x.size())
x = up3(x, x1)
print(x.size())

In [ ]:
n_classes = 2
outc = (OutConv_1D(32, n_classes))
logits = outc(x)

In [ ]:
logits.shape

---

# `UNet_1D` class test

In [1]:
from unet.unet_model import *
x = torch.arange(1, 365).float()
x = x.view(1, 1, -1)
n_channels = 1
inc = (DoubleConv_1D(n_channels, 32))
x1 = inc(x)
print(x.shape, x1.shape)

torch.Size([1, 1, 364]) torch.Size([1, 32, 364])


In [2]:
model = UNet_1D(n_channels=1, n_classes=2, bilinear=False)
print(  f'Network:\n'
        f'\t{model.n_channels} input channels\n'
        f'\t{model.n_classes} output channels (classes)\n'
        f'\t{"Bilinear" if model.bilinear else "Transposed conv"} upscaling')

Network:
	1 input channels
	2 output channels (classes)
	Transposed conv upscaling


In [3]:
masks_pred = model(x)

In [4]:
masks_pred.shape

torch.Size([1, 2, 364])

---

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import torch.nn as nn
import torch

# 创建一个 LSTM 层
lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=2)

# 创建一个输入张量
input = torch.randn(5, 3, 10)

# 初始化隐藏状态和细胞状态
h0 = torch.randn(2, 3, 20)
c0 = torch.randn(2, 3, 20)

# 将输入传递给 LSTM 层
output, (hn, cn) = lstm(input, (h0, c0))

# 假设你的模型是 model，输入的维度是 in_dim
model = lstm  # 你的模型

# 创建一个 SummaryWriter 对象
writer = SummaryWriter()

# 将模型和输入添加到 SummaryWriter
writer.add_graph(model, input)

# 关闭 SummaryWriter
writer.close()

---

# LSTM Test

In [1]:
from unet.unet_model import *
device = 'cuda'
feature_len = 365*4
x = torch.arange(0, feature_len).float()
x = x.view(5, 1, -1).to(device)
x.shape

torch.Size([5, 1, 292])

In [2]:
inc = (DoubleConv_1D(1, 32)).to(device)
down1 = (Down_1D(32, 64)).to(device)
down2 = (Down_1D(64, 128)).to(device)
down3 = (Down_1D(128, 256 // 1)).to(device)

In [3]:
x1=inc(x)
x2=down1(x1)
x3=down2(x2)
x4=down3(x3)
x1.shape, x2.shape, x3.shape, x4.shape

(torch.Size([5, 32, 280]),
 torch.Size([5, 64, 128]),
 torch.Size([5, 128, 52]),
 torch.Size([5, 256, 14]))

In [4]:
input_data = x1

In [5]:
n_layers=1
hidden_dim=256
batch_size, seq_len, feature_len  = input_data.size(0), input_data.size(1), input_data.size(2)
batch_size, seq_len, feature_len

(5, 32, 280)

In [6]:
lstm = nn.LSTM(input_size=feature_len, hidden_size=hidden_dim, num_layers=n_layers, batch_first=True).to(device)
linear = nn.Sequential(nn.Linear(hidden_dim, feature_len), nn.Tanh()).to(device)

In [7]:
h_0 = torch.zeros(n_layers, batch_size, hidden_dim).to(device)
c_0 = torch.zeros(n_layers, batch_size, hidden_dim).to(device)
h_0.shape, c_0.shape

(torch.Size([1, 5, 256]), torch.Size([1, 5, 256]))

In [8]:
recurrent_features, hidden = lstm(input_data, (h_0, c_0))

In [9]:
recurrent_features.shape, hidden[0].shape, hidden[1].shape

(torch.Size([5, 32, 256]), torch.Size([1, 5, 256]), torch.Size([1, 5, 256]))

In [10]:
outputs = linear(recurrent_features.contiguous().view(batch_size*seq_len, hidden_dim))
outputs = outputs.view(batch_size, seq_len, feature_len)
outputs.shape

torch.Size([5, 32, 280])

In [11]:
up1 = (Up_1D(256, 128 // 1, bilinear=False)).to(device)
up2 = (Up_1D(128, 64 // 1, bilinear=False)).to(device)
up3 = (Up_1D(64, 32 // 1, bilinear=False)).to(device)
outc = (OutConv_1D(32, 2)).to(device)

---

# LSTM-Unet Generator

In [1]:
import torch.nn as nn
from unet.unet_model import *

In [2]:
device = 'cuda'
feature_len = 365*4
x = torch.arange(0, feature_len).float()
x = x.view(5, 1, -1)
x.shape

torch.Size([5, 1, 292])

In [3]:
batch_size, seq_len, feature_len  = x.size(0), x.size(1), x.size(2)

In [4]:
model = LSTMUnetGenerator(batch_size, seq_len, feature_len, kernel_size=7, n_layers=1, hidden_dim=256)

In [5]:
y = model(x)

torch.Size([5, 32, 292]) torch.Size([5, 64, 146]) torch.Size([5, 128, 73]) torch.Size([5, 256, 36])
torch.Size([5, 128, 72])
torch.Size([5, 64, 144])
torch.Size([5, 32, 288])
torch.Size([5, 1, 288])


In [6]:
y.shape

torch.Size([5, 1, 288])

---